In [ ]:
# ============================================================
# PROBLEM 2: TRANSFORMER FROM SCRATCH
# Complete coursework implementation for Google Colab
# ============================================================
#
# Core attention, multi-head attention and encoder blocks are
# implemented using tensor operations. nn.MultiheadAttention and
# nn.Transformer are NOT used.
#
# Datasets:
#   Task A: UCI HAR raw inertial signals (reuses Problem 1 data)
#   Task B: Tiny Shakespeare character-level language modelling
#
# Outputs are saved under ./results_problem2/
#
# Colab setup (run in a separate cell if needed):
# !pip install torch numpy pandas matplotlib scikit-learn
#
# IMPORTANT:
# 1. Run this file/notebook from top to bottom.
# 2. Set QUICK_RUN=True for a smoke test; use False for final runs.
# 3. The instructor's exact 3-token Q/K/V example was not included
#    in the question. The hand-checkable example below is explicit
#    and can be replaced with the instructor's supplied values.
# 4. Training outcomes depend on runtime, random seed and hardware.
#    Do not invent or copy results; use the generated CSV/figures.
# ============================================================


# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

import os
import math
import time
import copy
import random
import zipfile
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

#QUICK_RUN = True to check that everything executes. Change it to False for final experiments.
QUICK_RUN = False

if QUICK_RUN:
    SEEDS = [42]
    HAR_EPOCHS = 2
    LM_EPOCHS = 2
    HAR_BATCH_SIZE = 128
    LM_BATCH_SIZE = 32
    HAR_LIMIT_TRAIN = 1200
    HAR_LIMIT_VAL = 400
    HAR_LIMIT_TEST = 600
    LM_STEPS_PER_EPOCH = 30
    ABLATION_EPOCHS = 1
else:
    SEEDS = [11, 22, 33]
    HAR_EPOCHS = 20
    LM_EPOCHS = 12
    HAR_BATCH_SIZE = 128
    LM_BATCH_SIZE = 64
    HAR_LIMIT_TRAIN = None
    HAR_LIMIT_VAL = None
    HAR_LIMIT_TEST = None
    LM_STEPS_PER_EPOCH = 250
    ABLATION_EPOCHS = 3

# HAR Transformer
HAR_D_MODEL = 128
HAR_HEADS = 4
HAR_LAYERS = 2
HAR_D_FF = 512
HAR_DROPOUT = 0.1
HAR_PATCH_SIZE = 8
HAR_LR = 1e-4

# Tiny Shakespeare Transformer
CONTEXT_LENGTH = 128
LM_D_MODEL = 128
LM_HEADS = 4
LM_LAYERS = 4
LM_D_FF = 512
LM_DROPOUT = 0.1
LM_LR = 3e-4

STUDENT_ID = "2026AJ05499"  # update if required

RESULTS_DIR = "results_problem2"
os.makedirs(RESULTS_DIR, exist_ok=True)

HAR_URL = (
    "https://archive.ics.uci.edu/static/public/240/"
    "human+activity+recognition+using+smartphones.zip"
)
HAR_OUTER_ZIP = "UCI_HAR_Dataset.zip"
HAR_FOLDER = "UCI HAR Dataset"

SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)
SHAKESPEARE_FILE = "tinyshakespeare.txt"


# ============================================================
# 2. REPRODUCIBILITY AND UTILITIES
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def save_figure(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, filename), dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()


def append_result(row):
    """Append a run using exactly the assignment's required columns."""
    columns = [
        "problem", "model", "seed", "seq_len", "test_acc", "macro_f1",
        "test_mse", "params", "sec_per_epoch", "val_loss_nats",
        "bits_per_char", "notes"
    ]
    path = os.path.join(RESULTS_DIR, "results.csv")
    frame = pd.DataFrame([{c: row.get(c, "") for c in columns}], columns=columns)
    if os.path.exists(path):
        old = pd.read_csv(path)
        old = old.reindex(columns=columns)
        frame = pd.concat([old, frame], ignore_index=True)
    frame.to_csv(path, index=False)


def maybe_limit(X, y, limit, seed):
    if limit is None or len(X) <= limit:
        return X, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=limit, replace=False)
    return X[idx], y[idx]


# ============================================================
# 3. TASK 1: SCALED DOT-PRODUCT ATTENTION FROM SCRATCH
# ============================================================

def scaled_dot_product_attention(Q, K, V, mask=None, dropout_p=0.0, training=False):
    """
    Q: (batch, heads, query_length, head_dim)
    K: (batch, heads, key_length, head_dim)
    V: (batch, heads, key_length, value_dim)
    mask: broadcastable to (batch, heads, query_length, key_length).
          Boolean True means KEEP/attend; False means BLOCK.
    Returns:
      output:  (batch, heads, query_length, value_dim)
      weights: (batch, heads, query_length, key_length)
    """
    d_k = Q.size(-1)

    # (B,H,Tq,D) @ (B,H,D,Tk) -> (B,H,Tq,Tk)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        mask = mask.to(device=scores.device, dtype=torch.bool)
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)

    weights = torch.softmax(scores, dim=-1)

    if dropout_p > 0 and training:
        weights = F.dropout(weights, p=dropout_p)

    # (B,H,Tq,Tk) @ (B,H,Tk,Dv) -> (B,H,Tq,Dv)
    output = torch.matmul(weights, V)
    return output, weights


# Unit tests directly after the function.
def test_scaled_dot_product_attention():
    torch.manual_seed(7)
    Q = torch.randn(2, 3, 5, 8)
    K = torch.randn(2, 3, 5, 8)
    V = torch.randn(2, 3, 5, 8)

    output, weights = scaled_dot_product_attention(Q, K, V)

    # Every attention row sums to one.
    assert torch.allclose(
        weights.sum(dim=-1),
        torch.ones_like(weights.sum(dim=-1)),
        atol=1e-6
    )

    # Match PyTorch's reference implementation.
    reference = F.scaled_dot_product_attention(Q, K, V)
    assert torch.allclose(output, reference, atol=1e-5, rtol=1e-5)

    # Causal mask: a token cannot attend to later tokens.
    T = 5
    causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
    _, causal_weights = scaled_dot_product_attention(Q, K, V, causal)
    for query_position in range(T):
        assert torch.all(causal_weights[..., query_position, query_position + 1:] == 0)

    print("Scaled dot-product attention tests PASSED.")


def hand_checkable_three_token_example():
    """
    A fully hand-checkable 3-token example.
    Replace Q/K/V below with the instructor's exact values if different.
    All vectors have d_k=2; V is chosen for easy arithmetic.
    """
    Q = torch.tensor([[[[1.0, 0.0],
                        [0.0, 1.0],
                        [1.0, 1.0]]]])
    K = torch.tensor([[[[1.0, 0.0],
                        [0.0, 1.0],
                        [1.0, 1.0]]]])
    V = torch.tensor([[[[1.0, 0.0],
                        [0.0, 2.0],
                        [2.0, 1.0]]]])

    output, weights = scaled_dot_product_attention(Q, K, V)
    print("\nThree-token hand-checkable example")
    print("Q =\n", Q[0, 0])
    print("K =\n", K[0, 0])
    print("V =\n", V[0, 0])
    print("Attention weights =\n", weights[0, 0])
    print("Output =\n", output[0, 0])

    # Independent explicit calculation for token 1:
    # scores = [1,0,1]/sqrt(2), weights = softmax(scores)
    scores_1 = torch.tensor([1.0, 0.0, 1.0]) / math.sqrt(2)
    weights_1 = torch.softmax(scores_1, dim=0)
    expected_1 = weights_1 @ V[0, 0]
    assert torch.allclose(output[0, 0, 0], expected_1, atol=1e-6)
    return output, weights


# ============================================================
# 4. TASK 2: MULTI-HEAD ATTENTION
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=128, num_heads=4, dropout=0.1):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # W_Q, W_K, W_V and W_O are learned projections.
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.attn_dropout = dropout
        self.output_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, return_attention=False):
        # x: (B,T,d_model)
        B, T, C = x.shape

        # Linear projections: (B,T,C) -> (B,T,C)
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # Split channels into heads:
        # (B,T,C) -> (B,T,H,Dh) -> (B,H,T,Dh)
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        output, weights = scaled_dot_product_attention(
            Q, K, V, mask,
            dropout_p=self.attn_dropout,
            training=self.training
        )

        # Merge heads: (B,H,T,Dh) -> (B,T,H,Dh) -> (B,T,C)
        output = output.transpose(1, 2).contiguous().view(B, T, C)
        output = self.output_dropout(self.W_O(output))

        if return_attention:
            return output, weights
        return output


# Unit test directly after the function/class it tests.
def test_multi_head_attention():
    torch.manual_seed(1)
    layer = MultiHeadAttention(32, 4, dropout=0.0)
    x = torch.randn(2, 7, 32)
    y, weights = layer(x, return_attention=True)
    assert y.shape == (2, 7, 32)
    assert weights.shape == (2, 4, 7, 7)
    assert torch.allclose(weights.sum(-1), torch.ones(2, 4, 7), atol=1e-6)
    print("Multi-head attention shape tests PASSED.")


# ============================================================
# 5. POSITIONAL ENCODING
# ============================================================

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term[:pe[0, :, 1::2].shape[-1]])
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class PositionalEncoding(nn.Module):
    def __init__(self, mode, d_model, max_len=4096):
        super().__init__()
        self.mode = mode
        if mode == "sinusoidal":
            self.encoding = SinusoidalPositionalEncoding(d_model, max_len)
        elif mode == "learned":
            self.position_embedding = nn.Embedding(max_len, d_model)
        elif mode == "none":
            self.encoding = nn.Identity()
        else:
            raise ValueError("mode must be none, sinusoidal or learned")

    def forward(self, x):
        if self.mode == "learned":
            positions = torch.arange(x.size(1), device=x.device)
            return x + self.position_embedding(positions).unsqueeze(0)
        return self.encoding(x)


# ============================================================
# 6. PRE-LAYERNORM ENCODER BLOCK
# ============================================================

class EncoderBlock(nn.Module):
    def __init__(self, d_model=128, num_heads=4, d_ff=512, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.residual_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, return_attention=False):
        # Pre-LN attention sub-layer and residual.
        if return_attention:
            attn_out, weights = self.attention(
                self.norm1(x), mask, return_attention=True
            )
        else:
            attn_out = self.attention(self.norm1(x), mask)

        x = x + self.residual_dropout(attn_out)

        # Pre-LN feed-forward sub-layer and residual.
        x = x + self.residual_dropout(self.feed_forward(self.norm2(x)))

        if return_attention:
            return x, weights
        return x


# Unit test directly after encoder implementation.
def test_encoder_block():
    block = EncoderBlock(128, 4, 512, 0.1)
    x = torch.randn(3, 20, 128)
    y = block(x)
    assert y.shape == x.shape
    print("Encoder block shape test PASSED.")


def verify_encoder_parameter_count():
    """
    Weight-only count per block:
      attention: 4*d*d
      FFN:       2*d*d_ff
    Biases and LayerNorm affine parameters are additional.
    """
    d, d_ff = 128, 512
    block = EncoderBlock(d, 4, d_ff, 0.1)
    weight_only = 4 * d * d + 2 * d * d_ff
    actual = sum(p.numel() for name, p in block.named_parameters()
                 if "weight" in name and p.ndim == 2)
    total = count_parameters(block)
    print("\nEncoder block parameter accounting")
    print("Formula weight-only:", weight_only)
    print("Actual matrix weights:", actual)
    print("All trainable parameters (including biases/LN):", total)
    # Matrix weights match the formula exactly.
    assert actual == weight_only
    assert total > weight_only
    return weight_only, total


# ============================================================
# 7. COMMON HAR DATA LOADING (SAME RAW DATA AS PROBLEM 1)
# ============================================================

HAR_CHANNELS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

ACTIVITY_NAMES = [
    "Walking", "Walking upstairs", "Walking downstairs",
    "Sitting", "Standing", "Laying"
]


def download_har_dataset():
    required = os.path.join(
        HAR_FOLDER, "train", "Inertial Signals", "body_acc_x_train.txt"
    )
    if os.path.isfile(required):
        print("UCI HAR dataset already available.")
        return

    if not os.path.exists(HAR_OUTER_ZIP):
        print("Downloading UCI HAR archive...")
        urllib.request.urlretrieve(HAR_URL, HAR_OUTER_ZIP)

    if not zipfile.is_zipfile(HAR_OUTER_ZIP):
        raise RuntimeError("Downloaded HAR archive is not a valid ZIP.")

    with zipfile.ZipFile(HAR_OUTER_ZIP) as zf:
        zf.extractall(".")

    # UCI's outer archive may contain a second ZIP.
    if not os.path.isdir(HAR_FOLDER):
        nested = None
        for root, _, files in os.walk("."):
            for name in files:
                candidate = os.path.join(root, name)
                if name.lower().endswith(".zip") and os.path.abspath(candidate) != os.path.abspath(HAR_OUTER_ZIP):
                    if zipfile.is_zipfile(candidate):
                        nested = candidate
                        break
            if nested:
                break
        if nested is None:
            raise FileNotFoundError("Could not find nested UCI HAR ZIP.")
        print("Extracting nested archive:", nested)
        with zipfile.ZipFile(nested) as zf:
            zf.extractall(".")

    if not os.path.isfile(required):
        raise FileNotFoundError("HAR extraction incomplete: " + required)


def load_har_split(split):
    base = os.path.join(HAR_FOLDER, split, "Inertial Signals")
    channels = []
    for channel in HAR_CHANNELS:
        path = os.path.join(base, f"{channel}_{split}.txt")
        channels.append(np.loadtxt(path, dtype=np.float32))
    X = np.stack(channels, axis=-1).astype(np.float32)  # (N,128,9)
    y = np.loadtxt(
        os.path.join(HAR_FOLDER, split, f"y_{split}.txt"), dtype=np.int64
    ) - 1
    subjects = np.loadtxt(
        os.path.join(HAR_FOLDER, split, f"subject_{split}.txt"), dtype=np.int64
    )
    return X, y, subjects


class HARSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def prepare_har_data(seed=42):
    download_har_dataset()
    X_train_all, y_train_all, subject_train = load_har_split("train")
    X_test, y_test, subject_test = load_har_split("test")

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    train_idx, val_idx = next(
        splitter.split(X_train_all, y_train_all, groups=subject_train)
    )

    X_train, y_train = X_train_all[train_idx], y_train_all[train_idx]
    X_val, y_val = X_train_all[val_idx], y_train_all[val_idx]
    train_subjects = subject_train[train_idx]
    val_subjects = subject_train[val_idx]

    assert set(train_subjects).isdisjoint(set(val_subjects))
    assert set(train_subjects).isdisjoint(set(subject_test))
    assert set(val_subjects).isdisjoint(set(subject_test))

    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    std[std < 1e-8] = 1.0

    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    X_test = (X_test - mean) / std

    X_train, y_train = maybe_limit(X_train, y_train, HAR_LIMIT_TRAIN, seed)
    X_val, y_val = maybe_limit(X_val, y_val, HAR_LIMIT_VAL, seed + 1)
    X_test, y_test = maybe_limit(X_test, y_test, HAR_LIMIT_TEST, seed + 2)

    loaders = {
        "train": DataLoader(HARSequenceDataset(X_train, y_train),
                            batch_size=HAR_BATCH_SIZE, shuffle=True),
        "val": DataLoader(HARSequenceDataset(X_val, y_val),
                          batch_size=HAR_BATCH_SIZE, shuffle=False),
        "test": DataLoader(HARSequenceDataset(X_test, y_test),
                           batch_size=HAR_BATCH_SIZE, shuffle=False)
    }
    print("HAR shapes:", X_train.shape, X_val.shape, X_test.shape)
    return loaders


# ============================================================
# 8. TASK A: HAR TRANSFORMER CLASSIFIER
# ============================================================

class HARTransformerClassifier(nn.Module):
    def __init__(
        self, input_features=9, patch_size=8, d_model=128,
        num_heads=4, num_layers=2, d_ff=512, dropout=0.1,
        positional_mode="sinusoidal"
    ):
        super().__init__()
        self.patch_size = patch_size
        self.patch_embedding = nn.Linear(input_features * patch_size, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.position = PositionalEncoding(
            positional_mode, d_model, max_len=256
        )
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, 6)
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, x, return_attention=False):
        # x: (B,128,9); split into 16 patches of 8 time steps.
        B, T, C = x.shape
        if T % self.patch_size != 0:
            raise ValueError("Sequence length must be divisible by patch_size")
        x = x.reshape(B, T // self.patch_size, self.patch_size * C)
        x = self.patch_embedding(x)  # (B, number_of_patches, d_model)

        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # CLS is position 0.
        x = self.position(x)

        all_weights = []
        for block in self.blocks:
            if return_attention:
                x, weights = block(x, return_attention=True)
                all_weights.append(weights)
            else:
                x = block(x)

        cls_representation = self.final_norm(x[:, 0])
        logits = self.classifier(cls_representation)
        if return_attention:
            return logits, all_weights
        return logits


def evaluate_classifier(model, loader):
    model.eval()
    predictions, targets = [], []
    total_loss, total = 0.0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss = F.cross_entropy(logits, y)
            total_loss += loss.item() * len(y)
            total += len(y)
            predictions.extend(logits.argmax(1).cpu().numpy())
            targets.extend(y.cpu().numpy())
    return {
        "loss": total_loss / max(total, 1),
        "accuracy": accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, average="macro", zero_division=0),
        "predictions": np.asarray(predictions),
        "targets": np.asarray(targets)
    }


def train_har_transformer(loaders, positional_mode, seed):
    set_seed(seed)
    model = HARTransformerClassifier(
        patch_size=HAR_PATCH_SIZE,
        d_model=HAR_D_MODEL,
        num_heads=HAR_HEADS,
        num_layers=HAR_LAYERS,
        d_ff=HAR_D_FF,
        dropout=HAR_DROPOUT,
        positional_mode=positional_mode
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=HAR_LR)
    best_f1, best_state = -1.0, None
    epoch_times, train_losses, val_losses = [], [], []

    for epoch in range(HAR_EPOCHS):
        model.train()
        start = time.perf_counter()
        total_loss, total = 0.0, 0
        for X, y in loaders["train"]:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(X)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(y)
            total += len(y)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        epoch_times.append(time.perf_counter() - start)
        train_losses.append(total_loss / max(total, 1))

        val = evaluate_classifier(model, loaders["val"])
        val_losses.append(val["loss"])
        if val["macro_f1"] > best_f1:
            best_f1 = val["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"HAR Transformer pos={positional_mode} seed={seed} "
            f"epoch={epoch+1}/{HAR_EPOCHS} "
            f"train_loss={train_losses[-1]:.4f} "
            f"val_loss={val['loss']:.4f} val_F1={val['macro_f1']:.4f}"
        )

    model.load_state_dict(best_state)
    test = evaluate_classifier(model, loaders["test"])
    result = {
        "problem": "Problem2_TaskA",
        "model": f"Transformer_pos_{positional_mode}",
        "seed": seed,
        "seq_len": 128,
        "test_acc": test["accuracy"],
        "macro_f1": test["macro_f1"],
        "val_macro_f1": best_f1,
        "params": count_parameters(model),
        "sec_per_epoch": float(np.mean(epoch_times)),
        "notes": f"patch={HAR_PATCH_SIZE}; heads={HAR_HEADS}; layers={HAR_LAYERS}"
    }
    append_result(result)
    return model, result, train_losses, val_losses, test


def run_task_a():
    loaders = prepare_har_data()
    all_results = []
    best_model = None
    best_score = -1
    best_test = None
    curves = {}

    for mode in ["none", "sinusoidal", "learned"]:
        for seed in SEEDS:
            model, result, train_curve, val_curve, test = train_har_transformer(
                loaders, mode, seed
            )
            all_results.append(result)
            if mode == "sinusoidal" and seed == SEEDS[0]:
                curves["train"] = train_curve
                curves["val"] = val_curve
            # Select models using validation F1 only; do not select on test data.
            if result["val_macro_f1"] > best_score:
                best_score = result["val_macro_f1"]
                best_model = model
                best_test = test

    df = pd.DataFrame(all_results)
    df.to_csv(os.path.join(RESULTS_DIR, "taskA_har_transformer_results.csv"), index=False)
    summary = df.groupby("model").agg(
        accuracy_mean=("test_acc", "mean"),
        accuracy_std=("test_acc", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        parameters=("params", "first"),
        sec_per_epoch=("sec_per_epoch", "mean")
    ).reset_index()
    summary.to_csv(os.path.join(RESULTS_DIR, "taskA_har_summary.csv"), index=False)
    print("\nTASK A RESULTS\n", summary.to_string(index=False))

    plt.figure(figsize=(8, 5))
    plt.plot(curves.get("train", []), label="Train loss")
    plt.plot(curves.get("val", []), label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.title("Task A HAR Transformer Learning Curve (sinusoidal)")
    plt.legend()
    save_figure("taskA_har_learning_curve.png")

    return loaders, summary, best_model, best_test


# ============================================================
# 9. TASK B: TINY SHAKESPEARE DATA AND BATCHING
# ============================================================

def download_shakespeare():
    if not os.path.exists(SHAKESPEARE_FILE):
        print("Downloading Tiny Shakespeare...")
        urllib.request.urlretrieve(SHAKESPEARE_URL, SHAKESPEARE_FILE)
    text = Path(SHAKESPEARE_FILE).read_text(encoding="utf-8")
    if len(text) < 10000:
        raise RuntimeError("Tiny Shakespeare text file looks incomplete.")
    return text


class CharacterDataset:
    def __init__(self, text):
        chars = sorted(list(set(text)))
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}
        self.vocab_size = len(chars)
        self.data = torch.tensor([self.stoi[c] for c in text], dtype=torch.long)

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join(self.itos[int(i)] for i in ids)


def get_lm_batch(data, batch_size, context_length, device=DEVICE):
    max_start = len(data) - context_length - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i:i + context_length] for i in starts])
    y = torch.stack([data[i + 1:i + context_length + 1] for i in starts])
    return x.to(device), y.to(device)


# ============================================================
# 10. CAUSAL TRANSFORMER LANGUAGE MODEL
# ============================================================

class CausalTransformerLM(nn.Module):
    def __init__(
        self, vocab_size, context_length=128, d_model=128,
        num_heads=4, num_layers=4, d_ff=512, dropout=0.1,
        use_causal_mask=True
    ):
        super().__init__()
        self.context_length = context_length
        self.use_causal_mask = use_causal_mask
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(context_length, d_model)
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying reduces parameters and is common in language models.
        self.lm_head.weight = self.token_embedding.weight
        self.dropout = nn.Dropout(dropout)

    def forward(self, idx, targets=None, return_attention=False):
        # idx: (B,T), integer character IDs.
        B, T = idx.shape
        if T > self.context_length:
            raise ValueError("Input exceeds configured context length")
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)[None, :, :]
        x = self.dropout(x)

        mask = None
        if self.use_causal_mask:
            # (T,T), True permits attention to current/past positions.
            mask = torch.tril(torch.ones(T, T, device=idx.device, dtype=torch.bool))

        attention_maps = []
        for block in self.blocks:
            if return_attention:
                x, weights = block(x, mask, return_attention=True)
                attention_maps.append(weights)
            else:
                x = block(x, mask)

        logits = self.lm_head(self.final_norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        if return_attention:
            return logits, loss, attention_maps
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=20):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                k = min(top_k, logits.size(-1))
                values, _ = torch.topk(logits, k)
                cutoff = values[:, [-1]]
                logits = logits.masked_fill(logits < cutoff, float("-inf"))

            probabilities = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probabilities, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


# Unit tests directly after causal model implementation.
def test_causal_language_model():
    model = CausalTransformerLM(
        vocab_size=17, context_length=16, d_model=32,
        num_heads=4, num_layers=2, d_ff=64, dropout=0.0,
        use_causal_mask=True
    ).to(DEVICE)
    x = torch.randint(0, 17, (2, 8), device=DEVICE)
    logits, loss, maps = model(x, x, return_attention=True)
    assert logits.shape == (2, 8, 17)
    assert loss.ndim == 0
    for weights in maps:
        for t in range(8):
            assert torch.all(weights[..., t, t+1:] == 0)
    print("Causal language model tests PASSED.")


def estimate_lm_loss(model, data, batch_size, context_length, batches=40):
    model.eval()
    losses = []
    with torch.no_grad():
        for _ in range(batches):
            x, y = get_lm_batch(data, batch_size, context_length)
            _, loss = model(x, y)
            losses.append(loss.item())
    return float(np.mean(losses))


def train_language_model(
    text_dataset, seed, use_causal_mask=True, context_length=CONTEXT_LENGTH,
    heads=LM_HEADS, epochs=LM_EPOCHS, steps_per_epoch=LM_STEPS_PER_EPOCH,
    tag="main"
):
    set_seed(seed)
    data = text_dataset.data
    split = int(0.90 * len(data))
    train_data, val_data = data[:split], data[split:]

    model = CausalTransformerLM(
        vocab_size=text_dataset.vocab_size,
        context_length=context_length,
        d_model=LM_D_MODEL,
        num_heads=heads,
        num_layers=LM_LAYERS,
        d_ff=LM_D_FF,
        dropout=LM_DROPOUT,
        use_causal_mask=use_causal_mask
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LM_LR)
    train_curve, val_curve, epoch_times = [], [], []

    for epoch in range(epochs):
        model.train()
        start = time.perf_counter()
        batch_losses = []

        for _ in range(steps_per_epoch):
            x, y = get_lm_batch(train_data, LM_BATCH_SIZE, context_length)
            optimizer.zero_grad(set_to_none=True)
            _, loss = model(x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        epoch_times.append(time.perf_counter() - start)
        train_loss = float(np.mean(batch_losses))
        val_loss = estimate_lm_loss(
            model, val_data, LM_BATCH_SIZE, context_length,
            batches=20 if QUICK_RUN else 50
        )
        train_curve.append(train_loss)
        val_curve.append(val_loss)

        print(
            f"LM {tag} seed={seed} epoch={epoch+1}/{epochs} "
            f"train={train_loss:.4f} val={val_loss:.4f} "
            f"BPC={val_loss / math.log(2):.4f}"
        )

    final_val = estimate_lm_loss(
        model, val_data, LM_BATCH_SIZE, context_length,
        batches=50 if not QUICK_RUN else 20
    )
    result = {
        "problem": "Problem2_TaskB",
        "model": f"Transformer_LM_{tag}",
        "seed": seed,
        "seq_len": context_length,
        "params": count_parameters(model),
        "sec_per_epoch": float(np.mean(epoch_times)),
        "val_loss_nats": final_val,
        "bits_per_char": final_val / math.log(2),
        "notes": f"heads={heads}; layers={LM_LAYERS}; causal_mask={use_causal_mask}"
    }
    append_result(result)
    return model, result, train_curve, val_curve, val_data


def save_generated_samples(model, dataset, val_data, filename="generated_samples.txt"):
    # Start generation from a short validation prefix.
    prompt = val_data[:1].view(1, -1).to(DEVICE)
    prompt = prompt[:, :min(40, prompt.size(1))]
    outputs = []
    for temperature in [0.2, 0.8, 1.5]:
        generated = model.generate(
            prompt.clone(), max_new_tokens=500,
            temperature=temperature, top_k=20
        )[0].tolist()
        sample = dataset.decode(generated[-500:])
        outputs.append((temperature, sample))

    path = os.path.join(RESULTS_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        for temperature, sample in outputs:
            f.write(f"===== TEMPERATURE {temperature} (500 CHARACTERS) =====\n")
            f.write(sample[:500])
            f.write("\n\n")
    print("Generated samples saved:", path)
    return outputs


def plot_lm_learning_curve(train_curve, val_curve):
    plt.figure(figsize=(8, 5))
    plt.plot(train_curve, label="Training loss")
    plt.plot(val_curve, label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss (nats)")
    plt.title("Tiny Shakespeare Transformer Learning Curve")
    plt.legend()
    save_figure("taskB_language_model_learning_curve.png")


# ============================================================
# 11. ANALYSIS: ATTENTION HEATMAPS FOR TWO HEADS
# ============================================================

def plot_attention_heatmaps(model, dataset, val_data):
    model.eval()
    # Decode a short validation sentence (or a contiguous 24-char segment).
    ids = val_data[:24].view(1, -1).to(DEVICE)
    with torch.no_grad():
        _, _, maps = model(ids, return_attention=True)

    # Last encoder layer; maps shape (B,heads,T,T).
    weights = maps[-1][0].detach().cpu().numpy()
    labels = list(dataset.decode(ids[0].cpu().tolist()))

    for head in [0, min(1, weights.shape[0] - 1)]:
        plt.figure(figsize=(8, 6))
        plt.imshow(weights[head], aspect="auto", cmap="viridis")
        plt.colorbar(label="Attention weight")
        plt.xticks(range(len(labels)), labels, rotation=90)
        plt.yticks(range(len(labels)), labels)
        plt.xlabel("Key character")
        plt.ylabel("Query character")
        plt.title(f"Last-layer attention heatmap: head {head + 1}")
        save_figure(f"taskB_attention_head_{head + 1}.png")


# ============================================================
# 12. HEAD-COUNT ABLATION: 1, 4, 8 HEADS AT FIXED d
# ============================================================

def run_head_ablation(dataset):
    rows = []
    epochs = ABLATION_EPOCHS if QUICK_RUN else max(3, ABLATION_EPOCHS)
    for heads in [1, 4, 8]:
        model, result, tr, va, _ = train_language_model(
            dataset, seed=SEEDS[0], use_causal_mask=True,
            context_length=CONTEXT_LENGTH, heads=heads,
            epochs=epochs, steps_per_epoch=LM_STEPS_PER_EPOCH,
            tag=f"heads_{heads}"
        )
        rows.append({
            "heads": heads,
            "val_loss_nats": result["val_loss_nats"],
            "bits_per_char": result["bits_per_char"],
            "parameters": result["params"],
            "sec_per_epoch": result["sec_per_epoch"]
        })

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(RESULTS_DIR, "taskB_heads_ablation.csv"), index=False)

    plt.figure(figsize=(7, 5))
    plt.plot(df["heads"], df["val_loss_nats"], marker="o")
    plt.xlabel("Number of attention heads (fixed d=128)")
    plt.ylabel("Validation loss (nats)")
    plt.title("Attention Head Count Ablation")
    save_figure("taskB_heads_ablation.png")
    print("\nHead ablation:\n", df.to_string(index=False))
    return df


# ============================================================
# 13. REMOVE CAUSAL MASK ABLATION
# ============================================================

def run_no_mask_ablation(dataset):
    rows = []
    for use_mask in [True, False]:
        model, result, train_curve, val_curve, val_data = train_language_model(
            dataset, seed=SEEDS[0], use_causal_mask=use_mask,
            context_length=CONTEXT_LENGTH, heads=LM_HEADS,
            epochs=ABLATION_EPOCHS if QUICK_RUN else max(3, ABLATION_EPOCHS),
            steps_per_epoch=LM_STEPS_PER_EPOCH,
            tag="causal_mask" if use_mask else "no_causal_mask"
        )
        rows.append({
            "causal_mask": use_mask,
            "val_loss_nats": result["val_loss_nats"],
            "bits_per_char": result["bits_per_char"]
        })

        plt.plot(
            val_curve,
            marker="o",
            label="Causal mask" if use_mask else "No causal mask"
        )

        if not use_mask:
            # Generate from the no-mask model as evidence of leakage.
            generated = model.generate(
                val_data[:1].view(1, -1)[:, :20].to(DEVICE),
                max_new_tokens=200, temperature=0.8, top_k=20
            )[0].tolist()
            with open(os.path.join(RESULTS_DIR, "no_mask_generated_sample.txt"),
                      "w", encoding="utf-8") as f:
                f.write(dataset.decode(generated))

    plt.xlabel("Epoch")
    plt.ylabel("Validation loss (nats)")
    plt.title("Causal Mask Ablation")
    plt.legend()
    save_figure("taskB_no_mask_result.png")

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(RESULTS_DIR, "taskB_no_mask_ablation.csv"), index=False)
    print("\nCausal mask ablation:\n", df.to_string(index=False))
    return df


# ============================================================
# 14. CONTEXT LENGTH TIME AND MEMORY SCALING
# ============================================================

def measure_context_scaling():
    rows = []
    batch_size = 8 if QUICK_RUN else 16
    for context in [64, 128, 256, 512]:
        model = CausalTransformerLM(
            vocab_size=65,
            context_length=context,
            d_model=LM_D_MODEL,
            num_heads=LM_HEADS,
            num_layers=LM_LAYERS,
            d_ff=LM_D_FF,
            dropout=0.0,
            use_causal_mask=True
        ).to(DEVICE)
        x = torch.randint(0, 65, (batch_size, context), device=DEVICE)
        y = torch.randint(0, 65, (batch_size, context), device=DEVICE)

        # Warm-up.
        model.train()
        for _ in range(2):
            logits, loss = model(x, y)
            loss.backward()
            model.zero_grad(set_to_none=True)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        timings = []
        for _ in range(5 if QUICK_RUN else 15):
            start = time.perf_counter()
            logits, loss = model(x, y)
            loss.backward()
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            timings.append(time.perf_counter() - start)
            model.zero_grad(set_to_none=True)

        peak_mb = (
            torch.cuda.max_memory_allocated() / (1024 ** 2)
            if DEVICE.type == "cuda" else float("nan")
        )
        rows.append({
            "context_length": context,
            "mean_step_seconds": float(np.mean(timings)),
            "peak_gpu_memory_mb": peak_mb,
            "attention_score_elements_per_layer": batch_size * LM_HEADS * context * context
        })
        del model, x, y
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(RESULTS_DIR, "taskB_context_scaling.csv"), index=False)

    plt.figure(figsize=(7, 5))
    plt.plot(df["context_length"], df["mean_step_seconds"], marker="o")
    plt.xlabel("Context length")
    plt.ylabel("Forward + backward time (seconds)")
    plt.title("Transformer Time vs Context Length")
    save_figure("taskB_time_vs_context.png")

    if DEVICE.type == "cuda":
        plt.figure(figsize=(7, 5))
        plt.plot(df["context_length"], df["peak_gpu_memory_mb"], marker="o")
        plt.xlabel("Context length")
        plt.ylabel("Peak allocated GPU memory (MB)")
        plt.title("Transformer Memory vs Context Length")
        save_figure("taskB_memory_vs_context.png")

    print("\nContext scaling:\n", df.to_string(index=False))
    return df


# ============================================================
# 15. OPTIONAL: COMPARE THE BEST PROBLEM 1 MODEL
# ============================================================

def load_problem1_summary_if_present():
    candidates = [
        "results/har_summary.csv",
        "results/har_summary.csv",
        "har_summary.csv"
    ]
    for path in candidates:
        if os.path.exists(path):
            print("\nFound Problem 1 HAR summary:", path)
            print(pd.read_csv(path).to_string(index=False))
            return pd.read_csv(path)
    print(
        "\nProblem 1 HAR summary not found. Place results/har_summary.csv "
        "from Problem 1 in the working directory to compare directly."
    )
    return None


# ============================================================
# 16. CREATE A SHORT ANALYSIS TEMPLATE FROM MEASURED RESULTS
# ============================================================

def write_analysis_template(task_a_summary=None, lm_result=None,
                            head_df=None, mask_df=None, context_df=None):
    path = os.path.join(RESULTS_DIR, "problem2_analysis_template.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("PROBLEM 2: TRANSFORMER FROM SCRATCH - ANALYSIS TEMPLATE\n\n")
        f.write("Use measured values from the CSV files and figures. Do not leave generic claims.\n\n")

        f.write("1. ATTENTION TESTS\n")
        f.write("- State that attention rows sum to one and compare the output with PyTorch SDPA.\n")
        f.write("- Explain the 3-token hand calculation using the instructor's exact Q/K/V values.\n")
        f.write("- Explain why the causal mask assigns zero probability to future positions.\n\n")

        f.write("2. ENCODER PARAMETER COUNT\n")
        f.write("- Weight-only count per block = 4*d^2 + 2*d*d_ff.\n")
        f.write("- At d=128, d_ff=512, weight-only count = 196,608.\n")
        f.write("- Add linear biases and LayerNorm affine parameters to obtain the total.\n\n")

        f.write("3. TASK A HAR CLASSIFIER\n")
        f.write("- Compare none, sinusoidal and learned positional encodings.\n")
        f.write("- Compare accuracy, macro-F1, parameter count and seconds per epoch.\n")
        f.write("- Compare with Problem 1 best RNN/LSTM/GRU result if its summary is available.\n\n")

        f.write("4. TASK B LANGUAGE MODEL\n")
        if lm_result is not None:
            f.write(f"- Validation loss (nats): {lm_result['val_loss_nats']:.5f}\n")
            f.write(f"- Bits per character: {lm_result['bits_per_char']:.5f}\n")
        f.write("- Describe the 0.2, 0.8 and 1.5 temperature samples.\n")
        f.write("- Lower temperature is more conservative; higher temperature is more diverse/noisy.\n\n")

        f.write("5. ABLATIONS\n")
        f.write("- Interpret both attention-head heatmaps.\n")
        f.write("- Compare 1, 4 and 8 heads at fixed d=128.\n")
        f.write("- Explain why removing the causal mask leaks future-character information.\n")
        f.write("- Discuss context-length timing and memory in relation to O(n^2*d).\n\n")

        if task_a_summary is not None:
            f.write("TASK A SUMMARY:\n")
            f.write(task_a_summary.to_string(index=False) + "\n\n")
        if head_df is not None:
            f.write("HEAD ABLATION:\n")
            f.write(head_df.to_string(index=False) + "\n\n")
        if mask_df is not None:
            f.write("MASK ABLATION:\n")
            f.write(mask_df.to_string(index=False) + "\n\n")
        if context_df is not None:
            f.write("CONTEXT SCALING:\n")
            f.write(context_df.to_string(index=False) + "\n\n")

    print("Analysis template saved:", path)


# ============================================================
# 17. MAIN
# ============================================================

def main():
    print("=" * 72)
    print("PROBLEM 2: TRANSFORMER FROM SCRATCH")
    print("=" * 72)
    print("Device:", DEVICE, "| QUICK_RUN:", QUICK_RUN)

    # Core correctness tests.
    test_scaled_dot_product_attention()
    hand_checkable_three_token_example()
    test_multi_head_attention()
    test_encoder_block()
    verify_encoder_parameter_count()
    test_causal_language_model()

    # Task A.
    print("\n" + "=" * 72)
    print("TASK A: HAR TRANSFORMER CLASSIFIER")
    print("=" * 72)
    har_loaders, task_a_summary, best_har_model, best_har_test = run_task_a()

    # Task B.
    print("\n" + "=" * 72)
    print("TASK B: TINY SHAKESPEARE LANGUAGE MODEL")
    print("=" * 72)
    raw_text = download_shakespeare()
    text_dataset = CharacterDataset(raw_text)
    print("Characters:", len(raw_text), "| Vocabulary:", text_dataset.vocab_size)
    print("Expected vocabulary is around 65; actual dataset vocabulary:", text_dataset.vocab_size)

    main_model, lm_result, train_curve, val_curve, val_data = train_language_model(
        text_dataset, seed=SEEDS[0], use_causal_mask=True,
        context_length=CONTEXT_LENGTH, heads=LM_HEADS,
        epochs=LM_EPOCHS, steps_per_epoch=LM_STEPS_PER_EPOCH,
        tag="main"
    )
    plot_lm_learning_curve(train_curve, val_curve)
    save_generated_samples(main_model, text_dataset, val_data)
    plot_attention_heatmaps(main_model, text_dataset, val_data)

    # Ablations and complexity measurements.
    print("\nRunning head-count ablation...")
    head_df = run_head_ablation(text_dataset)

    print("\nRunning causal-mask ablation...")
    mask_df = run_no_mask_ablation(text_dataset)

    print("\nMeasuring context-length scaling...")
    context_df = measure_context_scaling()

    # Compare to Problem 1 if its output CSV is present.
    load_problem1_summary_if_present()

    write_analysis_template(
        task_a_summary=task_a_summary,
        lm_result=lm_result,
        head_df=head_df,
        mask_df=mask_df,
        context_df=context_df
    )

    print("\n" + "=" * 72)
    print("PROBLEM 2 COMPLETED")
    print("=" * 72)
    print("All generated files:")
    for filename in sorted(os.listdir(RESULTS_DIR)):
        print("-", filename)
    print("\nRemember to include figures/tables in your 6-page report and refer")
    print("to every numbered figure/table in the report text.")


# ============================================================
# 18. RUN
# ============================================================

if __name__ == "__main__":
    main()


Device: cpu
PROBLEM 2: TRANSFORMER FROM SCRATCH
Device: cpu | QUICK_RUN: False
Scaled dot-product attention tests PASSED.

Three-token hand-checkable example
Q =
 tensor([[1., 0.],
        [0., 1.],
        [1., 1.]])
K =
 tensor([[1., 0.],
        [0., 1.],
        [1., 1.]])
V =
 tensor([[1., 0.],
        [0., 2.],
        [2., 1.]])
Attention weights =
 tensor([[0.4011, 0.1978, 0.4011],
        [0.1978, 0.4011, 0.4011],
        [0.2483, 0.2483, 0.5035]])
Output =
 tensor([[1.2033, 0.7967],
        [1.0000, 1.2033],
        [1.2552, 1.0000]])
Multi-head attention shape tests PASSED.
Encoder block shape test PASSED.

Encoder block parameter accounting
Formula weight-only: 196608
Actual matrix weights: 196608
All trainable parameters (including biases/LN): 198272
Causal language model tests PASSED.

TASK A: HAR TRANSFORMER CLASSIFIER
UCI HAR dataset already available.
HAR shapes: (5551, 128, 9) (1801, 128, 9) (2947, 128, 9)
HAR Transformer pos=none seed=11 epoch=1/20 train_loss=0.7379 